# Dev-AI Stage-1 simple-β — NB02: Primary OLS estimation + verdict

Pre-registered primary OLS of Y_p_logit on (X_lag_6 + X_lag_9 + X_lag_12 composite) with HAC standard errors; verdict-decision tree (PASS / FAIL / ESCALATE / SUBSTRATE_TOO_NOISY) emitted to `output/gate_verdict.json`.


### Why-markdown (4-part citation block)

**Reference.**

- Spec v1.0.2 at `contracts/docs/superpowers/specs/2026-05-04-dev-ai-stage-1-simple-beta-design.md` (file sha256 `d90f6302f9473aa938521ed0b7a9b58dc1c849cd74476cd90424f59f3bd3f37e`; decision_hash `7c72292516f58f3cf2d16464d4f84c3e7d7270ad2c5d3d8ed8fef6b3b2751f5a`).
- Plan v1.1.1 at `contracts/docs/superpowers/plans/2026-05-04-dev-ai-stage-1-simple-beta-implementation.md` (file sha256 `772b52e1f4b4e9e0ed964c3068b1948c24d5cfe27afc109e8e589a1ea790c036`).
- Pair D Phase 2 PASS verdict precedent (memory `project_pair_d_phase2_pass`): β_composite = +0.13670985, HAC SE = 0.02465, t = +5.5456, p_one = 1.46e-08; lag pattern β_6 / β_9 / β_12 = +/+/+; Pair D spec sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`.

**Why used.** NB02 is the verdict-bearing notebook of the dev-AI Stage-1 trio. Its primary OLS specification (Y_p_logit ~ X_lag_6 + X_lag_9 + X_lag_12 + controls; HAC SE) is pinned in spec §5.3, and its decision tree (PASS-trigger §3.1 / FAIL-trigger §3.2 / ESCALATE-trigger §3.3 / SUBSTRATE_TOO_NOISY §3.5 / N_MIN gate §3.6) is pinned in spec §3 + §8.

**Relevance to results.** This notebook emits `output/gate_verdict.json` — the machine-readable verdict that routes the iteration to {Stage-2 M-sketch dispatch | iteration-close | escalation to convex-payoff evidence | substrate-noise quarantine}. The β_composite point estimate, the HAC SE, the lag-pattern sign-row (β_6 / β_9 / β_12 ∈ {+, −}), and the p_one are the four binding numerics for verdict routing.

**Connection to simulator.** A PASS verdict here UNBLOCKS the Stage-2 ideal-scenario M sketch (per the active-iteration framework section in CLAUDE.md: "β confirmation unblocks M sketch; M sketch unblocks protocol design"). The Section-J Y_p PASS would inherit Pair D's precedent (Section G-T PASS) under spec §9.16 compositional-accounting risk; the R5 robustness arm (G-T minus J) is pre-authorized for v1.1 spec revision.


## Table of contents

NB02 owns the primary OLS + verdict-emission trios mapped to spec §5.3 / §5.2 / §3 / §8 (per plan Task 2.3):

- **§1 — Setup + panel re-load** (read panel_combined.parquet via `env.verify_panel()`; mirrors spec §4 sample-selection)
- **§2 — Trio: X-transform + composite construction** (mirrors spec §5.2 X construction: log(COP/USD) → demeaned + standardized lags 6/9/12 → equal-weight composite)
- **§3 — Trio: Control-variable specification** (mirrors spec §5.3 multicollinearity disposition + control-variable inclusion list)
- **§4 — Trio: Primary OLS fit + HAC SE + composite-β extraction** (mirrors spec §5.3 estimator)
- **§5 — Trio: Verdict-decision-tree evaluation** (mirrors spec §3 falsification criteria + §8.1 mapping rules; PASS / FAIL / ESCALATE / SUBSTRATE_TOO_NOISY / N_MIN-gate routing)
- **§6 — Trio: Numeric gate audit** (mirrors spec §3.6 N_MIN gate + §3.5 substrate-noise threshold; emits `output/gate_verdict.json` + `output/PRIMARY_RESULTS.md`)

Trio anatomy (per `feedback_notebook_trio_checkpoint`): `(why-markdown → code-cell → interpretation-markdown)` × N trios; HALT for orchestrator review after each trio.


## Trio-checkpoint discipline notice

This notebook is currently a **skeleton** (Phase 2 Task 2.1; scaffolding-only commit). Subsequent trios are authored ONE AT A TIME in Phase 2 Task 2.3 under HALT-for-review per `feedback_notebook_trio_checkpoint`; the orchestrator inspects the first-trio output before the second-trio dispatches per Plan Task 2.3 Step 0 prophylactic.

The Pair D v2.4 CORRECTIONS-β precedent (script-form deviation discovered post-execution) is the cautionary inheritance: this notebook MUST be authored in `.ipynb` form, NOT as a `.py` script. Verdict emission (`output/gate_verdict.json` + `output/PRIMARY_RESULTS.md`) is per spec §8 + plan Task 2.3.


## §1 — Trio 1: empalme-application VERIFICATION (not re-application)

### Why-markdown (4-part citation block)

**Reference.**

- Spec v1.0.2 §6 (methodology-break disposition; empalme factor primary; R1 2021 regime dummy alternative; FLAG-3 v1.0 closure auto-fallback ban; cell-pathology HALT-only fallbacks). File sha256 `d90f6302f9473aa938521ed0b7a9b58dc1c849cd74476cd90424f59f3bd3f37e`; decision_hash `7c72292516f58f3cf2d16464d4f84c3e7d7270ad2c5d3d8ed8fef6b3b2751f5a`. Path: `contracts/docs/superpowers/specs/2026-05-04-dev-ai-stage-1-simple-beta-design.md`.
- Plan v1.1.1 Task 2.3 Step 1 (empalme-application verification step preceding primary OLS). Plan file sha256 `772b52e1f4b4e9e0ed964c3068b1948c24d5cfe27afc109e8e589a1ea790c036`. Path: `contracts/docs/superpowers/plans/2026-05-04-dev-ai-stage-1-simple-beta-implementation.md`.
- Task 1.1 DE report Step 3 ("Y_p (Section J) construction with empalme + logit"); the empalme is **implicit in the FEX column choice** per DATA_PROVENANCE.md §2.A line 305 — DANE pre-applied empalme correction in `FEX_C` for the 2015-2020 catalogs (`era="empalme_2015_2020"`, 72 months, 2015-01 → 2020-12); `FEX_C18` is Marco-2018 native for 2021-01 → 2026-02 (62 months). No multiplicative factor is applied by consumers.
- DATA_PROVENANCE.md Task 1.1 §2.A (Y_p construction protocol, lines 300-316), §3.7 (panel_combined.parquet schema), §3.8 (panel sha256 `451f4c615c89a481da4ca132c79a55b04e00eecb9199746f544b22561ba0740d`).
- CORRECTIONS-α' commit `21beffade` (Marco-2005 → Marco-2018 disposition pin; window 2015-01 → 2026-03 because 2010-2014 Empalme files ship `RAMA4D` with CIIU Rev.3 codes per `feedback_schema_pre_flight_must_verify_values`).
- SPM D14 v1.0 closure (anomaly-HALT requirement; auto-fallback to (i) 3-month-rolling is anti-fishing-banned per spec §6 FLAG-3 closure).
- Pair D Phase 2 PASS verdict precedent (memory `project_pair_d_phase2_pass`): same Marco-2018 empalme produced a clean panel on Section G–T; PASS verdict β = +0.13670985 followed. Pair D spec sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`.

**Why used.** NB02's primary OLS specification regresses `Y_p_logit ~ X_lag6 + X_lag9 + X_lag12 + controls` on the empalme-applied Y_p panel (Trio 4 onward of NB02). The empalme factor is the spec §6 primary disposition for the Marco-2005 → Marco-2018 frame change at 2021-01; the alternative — R1 (2021 regime dummy) — serves as a sensitivity arm in NB03. This trio VERIFIES that the empalme application performed in Task 1.1 produced a Y_p panel without anomalies that would invalidate the primary specification — i.e., no Marco-2018-transition-induced spike at the 2020-12 → 2021-01 boundary, no residual segment-level mean-shift, no segment-specific variance regime change unaccounted for. Per the IMPORTANT clarification in the dispatch brief: Y_p_logit / Y_p_raw in `panel_combined.parquet` ARE ALREADY empalme-applied (FEX_C for pre-2021, FEX_C18 for post-2021); this trio's role is **VERIFY + surface anomalies**, NOT re-apply.

**Relevance to results.** A clean empalme produces a Y_p panel where the methodology-break is corrected at the level shift; an anomalous empalme would produce a discontinuity at 2020-12 → 2021-01 that biases β̂_composite either positively or negatively depending on the residual error direction. R1 (regime-dummy hedge) catches this in NB03; if R1 produces a sign-different β̂ from primary OLS, Clause-A FIRES per spec §6 v1.0.2 κ-tightening. This trio's empirical anomaly check is the FIRST line of defense — anomalies surfaced here HALT-and-surface to orchestrator (typed exception `DevAIStage1EmpalmeAnomalyPathological`) per spec §9.5 + §6 FLAG-3 closure. Auto-fallback to (i) 3-month-rolling is anti-fishing-banned; the disposition memo path enumerates ≥3 pivot options (fallback (i) / Y_s2 promotion / R1 promotion-to-primary) and the user picks → CORRECTIONS block → 3-way review.

**Connection to simulator.** Stage-2 M-sketch hedge calibration on a Section-J-referenced index requires a methodology-break-corrected Y series for stable variance/mean estimation. An anomalous empalme would mis-calibrate hedge strikes and produce a Stage-2 instrument that systematically over- or under-prices the methodology-break risk. Verification-then-proceed (Option A discipline) is required: PASS this trio → primary OLS (Trios 2-4 of NB02) → verdict (Trios 5-6 of NB02) → IF PASS verdict, Stage-2 M-sketch UNBLOCKED per the active-iteration framework (CLAUDE.md "β confirmation unblocks M sketch; M sketch unblocks protocol design"). FAIL this trio (anomaly fires) → no β̂ is interpretable until disposition memo resolves the empalme-residual-bias path.


In [ ]:
# Trio 1 — code cell: empalme-application VERIFICATION on already-applied Y_p panel
#
# Verifies that Task 1.1's empalme-application (FEX_C for 2015-2020 / FEX_C18 for 2021+)
# produced a Y_p panel without methodology-break anomalies. NOT re-application.
#
# Anomaly checks (per dispatch brief):
#   1. Boundary spike  : |Y[2021-01] - Y[2020-12]| > 3 * median within-segment |ΔY|
#   2. Mean-shift      : |mean(post-2021) - mean(pre-2021)| > 0.5 logit-units
#   3. Variance regime : std(post-2021) > 2 * std(pre-2021)
#   4. Segment sizes   : pre-2021 == 72 months (2015-01..2020-12); post-2021 == 62 months (2021-01..2026-02)
#
# If ANY flag fires, interpretation cell HALTs-and-surfaces per spec §9.5 + §6 FLAG-3.

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from env import verify_panel

# ── §1.1. Panel re-load via verify_panel() (asserts N=134, zero nulls, schema) ──
panel = verify_panel()
print(f"Panel loaded: N={len(panel)} rows; columns={list(panel.columns)}")

# Era split — Marco-2018 boundary at 2021-01-01 per DATA_PROVENANCE.md §2.A line 304
# `era` column is NOT in panel_combined.parquet (it lives only in upstream Y panels);
# we derive the segment label purely from year_month here.
boundary_ts = pd.Timestamp("2021-01-01", tz="UTC")
pre = panel[panel["year_month"] < boundary_ts].copy()
post = panel[panel["year_month"] >= boundary_ts].copy()
n_pre = len(pre)
n_post = len(post)
print(f"\nSegment sizes: pre-2021={n_pre} months ; post-2021={n_post} months ; total={n_pre + n_post}")

# ── §1.2. Per-segment summary statistics (Y_p_logit) ─────────────────────────
def _seg_stats(s: pd.Series) -> dict:
    return {
        "n": int(s.size),
        "mean": float(s.mean()),
        "std": float(s.std(ddof=1)),
        "min": float(s.min()),
        "max": float(s.max()),
    }

pre_stats = _seg_stats(pre["Y_p_logit"])
post_stats = _seg_stats(post["Y_p_logit"])
print("\nPre-2021 Y_p_logit:", pre_stats)
print("Post-2021 Y_p_logit:", post_stats)

# ── §1.3. Boundary-month differential ─────────────────────────────────────────
# Y_p_logit at 2021-01-31 minus Y_p_logit at 2020-12-31
y_2020_12 = panel.loc[panel["year_month"] == pd.Timestamp("2020-12-31", tz="UTC"), "Y_p_logit"]
y_2021_01 = panel.loc[panel["year_month"] == pd.Timestamp("2021-01-31", tz="UTC"), "Y_p_logit"]
assert len(y_2020_12) == 1 and len(y_2021_01) == 1, "boundary months must exist exactly once"
boundary_diff = float(y_2021_01.iloc[0] - y_2020_12.iloc[0])

# Within-segment month-to-month differentials (absolute)
pre_diffs = pre["Y_p_logit"].diff().dropna().abs()
post_diffs = post["Y_p_logit"].diff().dropna().abs()
median_pre_diff = float(pre_diffs.median())
median_post_diff = float(post_diffs.median())
median_within_diff = float(pd.concat([pre_diffs, post_diffs]).median())

print(f"\nBoundary-month differential Y_p_logit[2021-01] - Y_p_logit[2020-12] = {boundary_diff:+.6f}")
print(f"  pre-2021 median |ΔY| = {median_pre_diff:.6f}")
print(f"  post-2021 median |ΔY| = {median_post_diff:.6f}")
print(f"  pooled within-segment median |ΔY| = {median_within_diff:.6f}")
print(f"  3× envelope = {3.0 * median_within_diff:.6f}")

# ── §1.4. Anomaly flags (4 checks) ────────────────────────────────────────────
boundary_anomaly = bool(abs(boundary_diff) > 3.0 * median_within_diff)
mean_shift_anomaly = bool(abs(post_stats["mean"] - pre_stats["mean"]) > 0.5)
variance_regime_anomaly = bool(post_stats["std"] > 2.0 * pre_stats["std"])
segment_size_anomaly = bool((n_pre != 72) or (n_post != 62))

flags = {
    "boundary_anomaly": boundary_anomaly,
    "mean_shift_anomaly": mean_shift_anomaly,
    "variance_regime_anomaly": variance_regime_anomaly,
    "segment_size_anomaly": segment_size_anomaly,
}
any_flag = any(flags.values())
print("\nAnomaly flags:")
for k, v in flags.items():
    print(f"  {k}: {v}")
print(f"  ANY FLAG: {any_flag}")

# ── §1.5. Plot 1 — Y_p_logit time series with boundary line + segment shading ─
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(panel["year_month"], panel["Y_p_logit"], color="#1f4e79", linewidth=1.2, label="Y_p_logit")
ax.axvline(boundary_ts, color="#c00000", linestyle="--", linewidth=1.3,
           label="Marco-2018 boundary (2021-01-01)")
ax.axvspan(panel["year_month"].min(), boundary_ts, alpha=0.10, color="#1f4e79",
           label="pre-2021 (FEX_C empalme)")
ax.axvspan(boundary_ts, panel["year_month"].max(), alpha=0.10, color="#a36b00",
           label="post-2021 (FEX_C18 native)")
ax.set_title(
    "Y_p_logit (Section J share, logit) — empalme-applied panel; "
    f"boundary diff = {boundary_diff:+.4f}"
)
ax.set_xlabel("year_month")
ax.set_ylabel("Y_p_logit")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── §1.6. Plot 2 — Histogram of within-segment |ΔY| with boundary marked ──────
fig, ax = plt.subplots(figsize=(9, 4.0))
bins = np.linspace(0, max(pre_diffs.max(), post_diffs.max(), abs(boundary_diff)) * 1.05, 30)
ax.hist(pre_diffs, bins=bins, alpha=0.55, color="#1f4e79", label=f"pre-2021 |ΔY| (n={pre_diffs.size})")
ax.hist(post_diffs, bins=bins, alpha=0.55, color="#a36b00", label=f"post-2021 |ΔY| (n={post_diffs.size})")
ax.axvline(abs(boundary_diff), color="#c00000", linestyle="--", linewidth=1.6,
           label=f"|boundary diff| = {abs(boundary_diff):.4f}")
ax.axvline(3.0 * median_within_diff, color="#404040", linestyle=":", linewidth=1.2,
           label=f"3× envelope = {3.0 * median_within_diff:.4f}")
ax.set_title("Within-segment month-to-month |ΔY_p_logit| with boundary differential")
ax.set_xlabel("|ΔY_p_logit| (absolute month-to-month)")
ax.set_ylabel("count")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── §1.7. Segment summary table (markdown rendering) ──────────────────────────
summary_df = pd.DataFrame(
    {
        "pre-2021 (FEX_C)": [n_pre, pre_stats["mean"], pre_stats["std"],
                             pre_stats["min"], pre_stats["max"]],
        "post-2021 (FEX_C18)": [n_post, post_stats["mean"], post_stats["std"],
                                post_stats["min"], post_stats["max"]],
    },
    index=["N (months)", "mean(Y_p_logit)", "std(Y_p_logit)",
           "min(Y_p_logit)", "max(Y_p_logit)"],
)
print("\nSegment summary table:")
print(summary_df.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nBoundary differential (signed): {boundary_diff:+.6f}")
print(f"Mean-shift (post - pre): {post_stats['mean'] - pre_stats['mean']:+.6f} logit-units")
print(f"Variance ratio (post/pre std): {post_stats['std'] / pre_stats['std']:.4f}")

# ── §1.8. HALT-and-surface decision (interpretation cell elaborates) ──────────
if any_flag:
    print("\n[HALT-and-surface] At least one anomaly flag fired.")
    print("    Per spec §9.5 + §6 FLAG-3 closure: auto-fallback to (i) 3-month-rolling is anti-fishing-banned.")
    print("    Orchestrator must author disposition memo enumerating ≥3 pivot options:")
    print("      1. Fallback (i) 3-month-rolling on Y_p (HALT-disposition unconditional)")
    print("      2. Y_s2 (Section M) promotion to primary")
    print("      3. R1 (2021 regime dummy) promotion to primary specification")
    print("    User picks → CORRECTIONS block → 3-way review.")
else:
    print("\n[OK] No anomaly flags fired. Empalme-application verification PASSES.")
    print("    NB02 Trio 2 (primary OLS estimation per spec §5.3) dispatch unblocked.")


### Interpretation-markdown — Trio 1 readout

**Empalme application confirmation.** Per DATA_PROVENANCE.md §2.A line 305, the empalme is **implicit in the FEX column choice** — Task 1.1 used `FEX_C` (DANE empalme-corrected expansion factor; nota técnica `Nota-tecnica-empalme-series-GEIH.pdf` covers 2010-01 → 2020-12) for the 72 months in `era="empalme_2015_2020"` (2015-01 → 2020-12), and `FEX_C18` (Marco-2018 native) for the 62 months in `era="marco2018_native"` (2021-01 → 2026-02). Consumers do NOT multiply by an additional factor (raw share `Y_t = sum(FEX over Section-J young-employed) / sum(FEX over all young-employed)`, then logit-transformed per spec §5.1). The panel post-Task-1.1 is therefore empalme-applied per spec §6 primary disposition; this trio's role is verification not re-application.

**Boundary-month differential.** The signed value of `Y_p_logit[2021-01-31] - Y_p_logit[2020-12-31]` is printed by §1.3 above; the absolute value is compared to `3 × median(|ΔY|)` pooled across both segments. If `|boundary_diff| ≤ 3 × median_within_diff`, the boundary value sits within the routine within-segment month-to-month envelope and `boundary_anomaly` is `False`. If `|boundary_diff| > 3 × median_within_diff`, the methodology-break manifests as a discrete jump at 2021-01 that the FEX column swap did not absorb — `boundary_anomaly` fires.

**Mean-shift check.** The signed `mean(post-2021) - mean(pre-2021)` (in logit-units) is the second diagnostic. Spec §6 v1.0.2 acknowledges that the empalme correction may leave a residual level shift (the FEX rebasing does not guarantee a zero-mean structural transition between Marco-2005 and Marco-2018 frames; what the empalme guarantees is *expansion-factor scaling consistency*, not *raw-share invariance*). The criterion is `|mean(post) - mean(pre)| ≤ 0.5 logit-units`. A residual shift smaller than this magnitude is treated as routine within-empalme noise; a shift larger than 0.5 logit-units indicates the empalme has NOT fully neutralized the level shift and `mean_shift_anomaly` fires. Per spec §6 v1.0.2, the R1 (2021 regime dummy) and R3 (raw-OLS) hedges in NB03 are designed to absorb any residual shift; their relevance is amplified if the shift is large.

**Variance regime check.** The ratio `std(post-2021) / std(pre-2021)` is the third diagnostic. The criterion is `std(post) ≤ 2 × std(pre)`. A post-2021 standard deviation up to 2× the pre-2021 standard deviation is consistent with single-regime monthly-share noise. A ratio above 2× indicates the post-2021 era has materially higher Y_p variance — possibly driven by Section J's growing noise sensitivity as `raw_share` narrowed (per CORRECTIONS-κ FLAG-A interaction recorded in DATA_PROVENANCE.md §6.A line 369, where the realized `[0.014, 0.031]` raw-share range produces logit-derivative magnitudes 3-7× larger than spec-anticipated; small raw-share noise maps to amplified logit-share noise). If the variance regime is large, the HAC SE band in primary OLS may be heteroskedastically driven by post-2021 observations, and `variance_regime_anomaly` fires.

**Segment-size check.** The pre-2021 segment must have exactly 72 months (2015-01 → 2020-12 inclusive) and the post-2021 segment must have exactly 62 months (2021-01 → 2026-02 inclusive); 72 + 62 = 134 = N. Any deviation triggers `segment_size_anomaly` (which would indicate either a row-drop or a year_month-tz handling bug introduced by the join in Task 1.3 — both of which the §3.6 panel integrity audit ruled out for the as-emitted parquet, but which this trio re-verifies with a fresh boundary slice).

**Anomaly accounting (4 flags).** All four flag values are printed by §1.4 and §1.8 above — `boundary_anomaly`, `mean_shift_anomaly`, `variance_regime_anomaly`, `segment_size_anomaly`. The verdict is the disjunction.

- **All four FALSE → OK readout.** Empalme application verification PASSES. The empalme-applied panel is clean per spec §6 primary disposition; no methodology-break anomaly. NB02 Trio 2 (primary OLS estimation per spec §5.3) dispatch is unblocked.

- **ANY flag TRUE → HALT-and-surface.** Per spec §9.5 + §6 FLAG-3 closure, the executing specialist surfaces a typed exception (`DevAIStage1EmpalmeAnomalyPathological`) to the Foreground Orchestrator. **Auto-fallback to (i) 3-month-rolling is anti-fishing-banned** per spec §6 FLAG-3 closure (the rolling average would mechanically smooth the boundary spike and disguise — not resolve — the empalme-residual-bias). The orchestrator must author a disposition memo enumerating ≥3 pivot options:
  1. **Fallback (i) 3-month-rolling on Y_p** (HALT-disposition unconditional per spec §6 line 262). Cost: temporal smoothing reduces effective N for HAC by smoothing-window correlation; risk of obscuring the boundary signal.
  2. **Fallback (ii) quarterly aggregation** — UNVIABLE per spec §6 (collapses N below N_MIN=75).
  3. **Y_s2 (Section M) promotion to primary** — Section M is empirically structurally similar to Section J in cell-count regime (~140-245 vs. ~94-267 per DATA_PROVENANCE.md §4.B line 322), with lower CV (0.119 vs 0.252 per §5.A line 348); promotion would reduce variance somewhat. Cost: re-narrative of the Stage-2 hedge target from Section J to Section M (different structural-transformation fingerprint).
  4. **R1 (2021 regime dummy) promotion-to-primary** — the spec §6 alternative becomes the primary specification; the dummy absorbs any residual mean-shift. Cost: lose 1 degree of freedom; the regime-dummy interpretation is a different causal narrative than the empalme-clean interpretation (and would require explicit narrative adjustment in MEMO.md).

  User picks → CORRECTIONS block → 3-way review of CORRECTIONS revision.

**Pair D contrast.** Pair D's same Marco-2018 empalme produced a clean panel (no boundary anomaly fired in Pair D Phase 2 NB01 trio 1; PASS verdict β = +0.13670985 followed). This iteration narrows from G-T to Section J — the empalme should still apply cleanly because `RAMA4D_R4` is value-content-stable per Pair D §9.10 within the 2015-01 → 2026-03 window (CORRECTIONS-α' commit `21beffade`). An anomaly-fire here would be a NEW finding for the dev-AI iteration, not Pair-D-inherited; it would interact with the §9.16 compositional-accounting risk (Section J ⊂ Pair D Section G–T) by suggesting the methodology-break has differential impact on the Section-J subset that the broader G–T aggregate had washed out. The R5 robustness arm (G-T minus J) pre-authorized for v1.1 spec revision becomes load-bearing in this contingency.

**Trio-checkpoint discipline.** This trio HALTS per `feedback_notebook_trio_checkpoint`. Orchestrator reviews the printed flag values + plots; if no anomaly, dispatches Trio 2 (Primary OLS estimation per spec §5.3). If any anomaly, surfaces typed exception + disposition memo + ≥3 pivot options to the user.


## §2 — Trio 2: primary OLS estimation + verdict-primitive emission

### Why-markdown (4-part citation block per `feedback_notebook_citation_block`)

**Reference.**

- Spec v1.0.2 (sha256 `d90f6302f9473aa938521ed0b7a9b58dc1c849cd74476cd90424f59f3bd3f37e`, decision_hash `7c72292516f58f3cf2d16464d4f84c3e7d7270ad2c5d3d8ed8fef6b3b2751f5a`, `contracts/docs/superpowers/specs/2026-05-04-dev-ai-stage-1-simple-beta-design.md`): §3 (estimator + hypothesis), §3.3 (verdict primitives PASS / FAIL / INDETERMINATE + Clause-A + Clause-B), §3.4 (HAC bandwidth `L = 12` unconditional Newey-West), §3.5 (composite β = β_6 + β_9 + β_12, **SUM** not average; linear-restriction `c = (0, 1, 1, 1)'` for composite SE), §3.6 (N gate, N_MIN = 75), §5.3 (multicollinearity acknowledgment + composite-SE deflation; design matrix `Y_p_logit ~ X_lag6 + X_lag9 + X_lag12 + intercept`), §6 (R1 + R3 hedges in NB03; CORRECTIONS-κ tightening makes them load-bearing), §9.16 (compositional-accounting risk Section J ⊂ Pair D Section G-T).
- Plan v1.1.1 (sha256 `772b52e1f4b4e9e0ed964c3068b1948c24d5cfe27afc109e8e589a1ea790c036`, `contracts/docs/superpowers/plans/2026-05-04-dev-ai-stage-1-simple-beta-implementation.md`): Task 2.3 Step 2 (primary OLS estimation, NB02 Trio 2).
- Pair D PASS verdict precedent — project memory `project_pair_d_phase2_pass`: β_composite = +0.13670985, HAC SE = 0.02465, t = +5.5456, p_one = 1.46e-08, lag pattern β_6/β_9/β_12 = +/+/+ with β_6 ≈ 80% of composite (RC FLAG #3 inherited into Stage-2 dispatch brief). Pair D spec sha256 `964c62cca0be1b9070944b5398fe97886c6d07d37ba7121199de8ccc341ef659`.
- Trio 4 NB01 multicollinearity context: spec §5.3 anticipates composite-SE deflation `SE_composite < sqrt(SE_β_6² + SE_β_9² + SE_β_12²)` due to negative covariance contributions among lag coefficients; this trio empirically confirms the deflation magnitude.
- Statsmodels OLS reference: `statsmodels.regression.linear_model.OLS.fit(cov_type='HAC', cov_kwds={'maxlags': L})` with `L = 12` per spec §3.4 unconditional pin (no Bartlett-bandwidth-selection rule allowed; bandwidth is fixed pre-estimation).
- One-sided p-value: `p_one = 1 - Φ(t_composite)` (large-N normal approximation; HAC asymptotic distribution is N(0,1)). Sign-positive expected per Baumol → US-Colombia tech-labor wage arbitrage → US tech-firm offshoring transmission chain.

**Why used.**

This trio executes the **primary OLS estimation** that produces the Stage-1 verdict-primitive (PASS / FAIL / INDETERMINATE per spec §3.3). The estimator is OLS of `Y_p_logit` on `(X_lag6, X_lag9, X_lag12)` plus intercept (134 obs post-lag-12 per spec §3.6 / `N_EXPECTED_POST_LAG_12 = 134` env constant). The hypothesis under test is one-sided `H1: β_composite > 0` where `β_composite ≡ β_6 + β_9 + β_12` per spec §3.5 (SUM not average; sign-positive expected). HAC standard errors with bandwidth `L = 12` per spec §3.4 unconditional Newey-West replace IID SEs to absorb serial correlation in monthly residuals. Verdict-primitive is read from `(t_composite, sign(β̂_composite), p_one_HAC)` against §3.3 thresholds: PASS = `t > 1.645 AND β̂ > 0 AND p_one < 0.05`; FAIL = `t ≤ 0 OR p_one > 0.10`; otherwise INDETERMINATE (routes to NB03 Clause-A R1+R3 sign-AGREE check + Clause-B HAC-bootstrap reconciliation in spec §3.4).

**Relevance to results.**

This trio is the **central computation** of Stage-1. PASS unblocks Stage-2 M-sketch authoring per framework ideal-scenario clause (Panoptic perpetual on Section-J-referenced index; hedge-strike calibration consumes both `β_composite` and the lag-pattern decomposition). FAIL routes to disposition memo per spec §3.3 (typed-exception → enumerated alternatives: Y_s1 EMS Section J income N=87 sensitivity arm promotion, Y_s2 Section M promotion, etc.). INDETERMINATE routes to convex-payoff escalation per spec §5.5 (quantile regression on Y, GARCH-X variance specification, lower-tail / extreme-value analysis) — testing convex-instrument fitness directly when mean-β is statistically ambiguous. Trio 2's printed numerics feed Phase-3 result memo §10 (verdict statement) + §11.X (Pair D contrast per spec §9.17(b)) verbatim. Per spec §6 v1.0.2 CORRECTIONS-κ, the verdict-primitive emitted here is **provisional** pending NB03 R1 + R3 sign-AGREE check; if R1 OR R3 produces a sign-different β̂ in NB03, Clause-A FIRES regardless of primary t-stat.

**Connection to simulator.**

Stage-2 M-sketch hedge-strike calibration on a Section-J-referenced index requires (a) `β_composite` (overall sensitivity of Section J share to lagged COP/USD shocks; calibrates payoff slope at the M-sketch reference price) and (b) the lag-pattern decomposition `(β_6, β_9, β_12)` (calibrates hedge-roll frequency: lag-6 concentration → shorter rolls; lag-12 concentration → longer rolls). The Pair D RC FLAG #3 inheritance is binding: `β_6 ≈ 80%` of composite in Pair D ⇒ short-roll bias in M-sketch hedge schedule. If this iteration's lag profile differs materially (e.g., flatter distribution across all three lags, or lag-12-dominant), M-sketch parameters must adjust accordingly — the dispatch brief into Stage-2 will document the realized lag-share decomposition for this purpose. Trio 2 emits both (composite + per-lag breakdown).


In [ ]:
# Trio 2 — code cell: primary OLS estimation per spec §5.3 + HAC SE per §3.4 + composite β per §3.5
#
# Estimator (spec §5.3):    Y_p_logit = α + β_6·X_lag6 + β_9·X_lag9 + β_12·X_lag12 + ε
# HAC bandwidth (spec §3.4): L = 12 (Newey-West, unconditional pin)
# Composite (spec §3.5):     β_composite = β_6 + β_9 + β_12 (SUM); c = (0, 1, 1, 1)'
# Verdict primitives (§3.3): PASS = t>1.645 AND β̂>0 AND p_one<0.05
#                            FAIL = t≤0 OR p_one>0.10
#                            else INDETERMINATE (routes to NB03 Clause-A + Clause-B)

from __future__ import annotations

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

from env import (
    verify_panel,
    PAIR_D_BETA_COMPOSITE,
    PAIR_D_HAC_SE,
    PAIR_D_T_STAT,
    PAIR_D_P_ONE,
    N_EXPECTED_POST_LAG_12,
)

# ── §2.1. Panel re-load via verify_panel() (re-asserts N=134 + zero nulls) ──
panel = verify_panel()
assert len(panel) == N_EXPECTED_POST_LAG_12, (
    f"panel row count {len(panel)} != expected {N_EXPECTED_POST_LAG_12} (spec §3.6)"
)
print(f"Panel loaded: N={len(panel)} rows; sample window "
      f"{panel['year_month'].min().strftime('%Y-%m')} → "
      f"{panel['year_month'].max().strftime('%Y-%m')}")

# ── §2.2. Design matrix construction (spec §5.3) ────────────────────────────
# Y = Y_p_logit (134 × 1); X = (X_lag6, X_lag9, X_lag12) plus intercept (134 × 4)
y = panel["Y_p_logit"].to_numpy(dtype=float)
X_no_const = panel[["X_lag6", "X_lag9", "X_lag12"]].to_numpy(dtype=float)
X = sm.add_constant(X_no_const, has_constant="add")  # 134 × 4 with intercept first
assert X.shape == (134, 4), f"design matrix shape {X.shape} != (134, 4)"
print(f"Design matrix: y shape={y.shape}; X shape={X.shape} "
      f"(columns: const, X_lag6, X_lag9, X_lag12)")

# ── §2.3. OLS fit with IID SE (sanity check) ─────────────────────────────────
model = sm.OLS(y, X)
result_iid = model.fit()
print("\n=== IID-SE sanity check (NOT used for verdict; for diagnostic only) ===")
print(f"  R²              = {result_iid.rsquared:.6f}")
print(f"  R²-adj          = {result_iid.rsquared_adj:.6f}")
print(f"  F-stat          = {result_iid.fvalue:.4f}  (p={result_iid.f_pvalue:.4e})")
print(f"  α (intercept)   = {result_iid.params[0]:+.8f}  (IID SE {result_iid.bse[0]:.6f})")
print(f"  β_6             = {result_iid.params[1]:+.8f}  (IID SE {result_iid.bse[1]:.6f})")
print(f"  β_9             = {result_iid.params[2]:+.8f}  (IID SE {result_iid.bse[2]:.6f})")
print(f"  β_12            = {result_iid.params[3]:+.8f}  (IID SE {result_iid.bse[3]:.6f})")

# ── §2.4. OLS fit with HAC SE (Newey-West, L=12 per spec §3.4) ──────────────
result_hac = model.fit(cov_type="HAC", cov_kwds={"maxlags": 12})
alpha_hat = float(result_hac.params[0])
beta_6 = float(result_hac.params[1])
beta_9 = float(result_hac.params[2])
beta_12 = float(result_hac.params[3])
se_alpha = float(result_hac.bse[0])
se_beta_6 = float(result_hac.bse[1])
se_beta_9 = float(result_hac.bse[2])
se_beta_12 = float(result_hac.bse[3])

print("\n=== HAC-SE primary-spec OLS (verdict-bearing per spec §5.3 + §3.4) ===")
print(f"  α (intercept)   = {alpha_hat:+.8f}  (HAC SE {se_alpha:.6f}; "
      f"t={alpha_hat/se_alpha:+.4f})")
print(f"  β_6             = {beta_6:+.8f}  (HAC SE {se_beta_6:.6f}; "
      f"t={beta_6/se_beta_6:+.4f})")
print(f"  β_9             = {beta_9:+.8f}  (HAC SE {se_beta_9:.6f}; "
      f"t={beta_9/se_beta_9:+.4f})")
print(f"  β_12            = {beta_12:+.8f}  (HAC SE {se_beta_12:.6f}; "
      f"t={beta_12/se_beta_12:+.4f})")

# ── §2.5. Composite β + composite HAC SE via linear restriction (spec §3.5) ──
# β_composite = c'β where c = (0, 1, 1, 1)'  (excludes intercept)
# Var(c'β̂) = c'Σ̂c where Σ̂ = HAC variance-covariance matrix
c = np.array([0.0, 1.0, 1.0, 1.0])
beta_composite = float(c @ result_hac.params)        # = β_6 + β_9 + β_12
cov_hac = np.asarray(result_hac.cov_params())
var_composite = float(c @ cov_hac @ c)
assert var_composite > 0, f"composite variance non-positive: {var_composite}"
se_composite = float(np.sqrt(var_composite))
t_composite = beta_composite / se_composite
# One-sided p-value (large-N normal): P(Z > t) = 1 - Φ(t)
p_one_hac = float(1.0 - stats.norm.cdf(t_composite))

print("\n=== Composite β (spec §3.5: β_composite = β_6 + β_9 + β_12 [SUM]) ===")
print(f"  β̂_composite        = {beta_composite:+.8f}")
print(f"  HAC SE_composite   = {se_composite:.8f}")
print(f"  t_composite        = {t_composite:+.6f}")
print(f"  p_one_HAC          = {p_one_hac:.6e}")

# ── §2.6. Composite-SE deflation diagnostic (spec §5.3 multicollinearity) ────
# Naive sum-variance ignoring covariance: sqrt(SE_6² + SE_9² + SE_12²)
naive_sum_var_se = float(np.sqrt(se_beta_6**2 + se_beta_9**2 + se_beta_12**2))
deflation_ratio = se_composite / naive_sum_var_se
print(f"\n=== Composite-SE deflation diagnostic (spec §5.3) ===")
print(f"  HAC SE_composite           = {se_composite:.6f}")
print(f"  naive-sum sqrt(ΣSE²)       = {naive_sum_var_se:.6f}")
print(f"  deflation ratio (HAC/naive)= {deflation_ratio:.4f}")
if deflation_ratio < 1.0:
    print(f"  CONFIRMED: SE_composite < naive-sum (covariance contributions are net-negative).")
else:
    print(f"  ANOMALOUS: SE_composite ≥ naive-sum; covariance contributions are net-positive "
          f"(re-check HAC fit; expected per spec §5.3 multicollinearity is deflation).")

# ── §2.7. Lag-pattern concentration shares (vs Pair D RC FLAG #3 lag-6 ≈ 80%) ─
if abs(beta_composite) > 1e-12:
    share_lag6 = beta_6 / beta_composite
    share_lag9 = beta_9 / beta_composite
    share_lag12 = beta_12 / beta_composite
else:
    share_lag6 = share_lag9 = share_lag12 = float("nan")
print("\n=== Lag-pattern concentration (spec §11.X(b) Pair D contrast) ===")
print(f"  β_6  share = {share_lag6:+.4f}  ({share_lag6*100:+.2f}% of composite)")
print(f"  β_9  share = {share_lag9:+.4f}  ({share_lag9*100:+.2f}% of composite)")
print(f"  β_12 share = {share_lag12:+.4f}  ({share_lag12*100:+.2f}% of composite)")
print(f"  Pair D reference: β_6 ≈ 80% of composite (RC FLAG #3 inheritance).")

# ── §2.8. Verdict-primitive determination per spec §3.3 ──────────────────────
sign_positive = beta_composite > 0
t_passes_one_sided = t_composite > 1.645
p_passes_strict = p_one_hac < 0.05
p_in_indeterminate_band = 0.05 <= p_one_hac <= 0.10
t_fails = t_composite <= 0
p_fails = p_one_hac > 0.10

if sign_positive and t_passes_one_sided and p_passes_strict:
    verdict_primitive = "PASS"
elif t_fails or p_fails:
    verdict_primitive = "FAIL"
else:
    verdict_primitive = "INDETERMINATE"

print("\n=== Verdict-primitive determination (spec §3.3) ===")
print(f"  sign(β̂_composite) > 0 ? {sign_positive}")
print(f"  t_composite > 1.645   ? {t_passes_one_sided}")
print(f"  p_one_hac < 0.05      ? {p_passes_strict}")
print(f"  t_composite ≤ 0       ? {t_fails}")
print(f"  p_one_hac > 0.10      ? {p_fails}")
print(f"  ----------------------------------")
print(f"  VERDICT-PRIMITIVE     : {verdict_primitive}")
print(f"  (PROVISIONAL — pending NB03 Clause-A R1+R3 sign-AGREE + Clause-B HAC-bootstrap)")

# ── §2.9. Pair D PASS contrast (informational) ───────────────────────────────
print("\n=== Pair D PASS verdict contrast (project memory project_pair_d_phase2_pass) ===")
print(f"  Pair D β_composite = {PAIR_D_BETA_COMPOSITE:+.8f}  (HAC SE {PAIR_D_HAC_SE:.5f}; "
      f"t={PAIR_D_T_STAT:+.4f}; p_one={PAIR_D_P_ONE:.2e})")
print(f"  Section J (this iter) Δβ_composite = "
      f"{beta_composite - PAIR_D_BETA_COMPOSITE:+.8f}")
print(f"  Section J (this iter) ratio        = "
      f"{beta_composite / PAIR_D_BETA_COMPOSITE:+.4f}× Pair D")

# ── §2.10. Plot 1 — bar chart of β_6/β_9/β_12 with HAC SE error bars ─────────
fig, ax = plt.subplots(figsize=(8.0, 4.5))
labels = ["β_6 (lag 6)", "β_9 (lag 9)", "β_12 (lag 12)"]
vals = [beta_6, beta_9, beta_12]
errs = [1.96 * se_beta_6, 1.96 * se_beta_9, 1.96 * se_beta_12]
colors = ["#1f4e79", "#a36b00", "#3a7d44"]
bars = ax.bar(labels, vals, yerr=errs, capsize=6,
              color=colors, alpha=0.85, edgecolor="#202020", linewidth=0.8)
ax.axhline(0, color="#404040", linewidth=0.6)
ax.axhline(beta_composite, color="#c00000", linestyle="--", linewidth=1.4,
           label=f"β̂_composite = {beta_composite:+.4f} (HAC SE {se_composite:.4f})")
ax.axhline(PAIR_D_BETA_COMPOSITE, color="#666666", linestyle=":", linewidth=1.0,
           label=f"Pair D β_composite = {PAIR_D_BETA_COMPOSITE:+.4f}")
ax.set_title(
    "Per-lag β̂ with 95% HAC CI error bars; composite (red dashed) vs Pair D (gray dotted)\n"
    f"Verdict-primitive: {verdict_primitive} (t={t_composite:+.3f}, p_one={p_one_hac:.3e})"
)
ax.set_ylabel("β̂ (logit-Y per unit log-COP/USD)")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# ── §2.11. Plot 2 — residuals vs year_month (homoskedasticity / autocorr eyeball)
resid = np.asarray(result_hac.resid)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(panel["year_month"], resid, color="#1f4e79", linewidth=1.0, marker="o",
        markersize=2.5, label="OLS residuals")
ax.axhline(0, color="#404040", linewidth=0.6)
ax.axvline(pd.Timestamp("2021-01-01", tz="UTC"), color="#c00000", linestyle="--",
           linewidth=1.0, label="Marco-2018 boundary (2021-01)")
ax.set_title(
    f"Primary OLS residuals over time (n={len(resid)}); "
    f"std={float(resid.std(ddof=1)):.4f}, mean={float(resid.mean()):+.4e}"
)
ax.set_xlabel("year_month")
ax.set_ylabel("residual (logit-Y units)")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── §2.12. Plot 3 — actual vs fitted Y_p_logit ───────────────────────────────
y_fitted = np.asarray(result_hac.fittedvalues)
fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.scatter(y_fitted, y, color="#1f4e79", alpha=0.65, s=28, edgecolor="#202020", linewidth=0.4)
yx_min = float(min(y.min(), y_fitted.min())) - 0.02
yx_max = float(max(y.max(), y_fitted.max())) + 0.02
ax.plot([yx_min, yx_max], [yx_min, yx_max], color="#c00000", linestyle="--",
        linewidth=1.2, label="45° (perfect fit)")
ax.set_xlim(yx_min, yx_max)
ax.set_ylim(yx_min, yx_max)
ax.set_title(f"Actual vs Fitted Y_p_logit (R²={result_iid.rsquared:.4f})")
ax.set_xlabel("Fitted Y_p_logit")
ax.set_ylabel("Actual Y_p_logit")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()


### Interpretation-markdown — Trio 2 readout

**Composite β estimate.** The primary OLS yields `β̂_composite` printed in §2.5 above. Per spec §3.5, this is the **SUM** `β_6 + β_9 + β_12` (NOT the average), constructed via the linear restriction `c'β̂` with `c = (0, 1, 1, 1)'` over the parameter vector `(α, β_6, β_9, β_12)'`. The Pair D PASS reference is `β_composite = +0.13670985` (project memory `project_pair_d_phase2_pass`); this iteration's printed `Δβ_composite` (§2.9) and the ratio-against-Pair-D quantify whether the Section-J narrowing produces a similar-magnitude or different signal vs the Pair D Section G–T broad-services aggregate. Per spec §9.16, the compositional-accounting overlap (Section J ⊂ Pair D Section G–T) means this iteration's β̂ is NOT an independent draw from Pair D's; the contrast is informational, not a methodological independence claim.

**Composite HAC SE + t-stat + p_one_HAC.** §2.5 prints `SE_composite`, `t_composite`, and `p_one_HAC`. Standard error is computed from the HAC variance-covariance matrix as `SE_composite = sqrt(c'Σ̂c)`, with `Σ̂` from `result_hac.cov_params()` under the spec §3.4 unconditional Newey-West bandwidth `L = 12`. The one-sided p-value uses the large-N normal approximation `p_one = 1 - Φ(t_composite)` (HAC asymptotic distribution of the standardized composite is N(0,1) under the null `H_0: β_composite = 0`). Sign-positive expected per the Baumol → US-Colombia tech-labor wage arbitrage → US tech-firm offshoring transmission chain documented in spec §1 + §2.

**Verdict-primitive determination.** Per spec §3.3, the primitive is one of {PASS, FAIL, INDETERMINATE}, computed in §2.8 from three boolean conditions:
- **PASS** ⇔ `sign(β̂_composite) > 0` AND `t_composite > 1.645` AND `p_one_HAC < 0.05`. All three must hold simultaneously. Action: routes to NB03 Clause-A R1+R3 sign-AGREE check (still required per CORRECTIONS-κ — primary PASS does NOT bypass robustness AGREE) + Clause-B HAC-bootstrap reconciliation; if both downstream gates clear, Stage-2 M-sketch authoring is unblocked per framework ideal-scenario clause.
- **FAIL** ⇔ `t_composite ≤ 0` OR `p_one_HAC > 0.10`. Either condition suffices. Action: typed-exception → disposition memo per spec §3.3 (enumerated alternatives include Y_s1 EMS-Section-J-income promotion, Y_s2 Section-M promotion, Y_s3 UNCTAD-EBOPS-9 panel promotion, R5 robustness arm authorization).
- **INDETERMINATE** ⇔ neither PASS nor FAIL — i.e., `0 < t_composite ≤ 1.645` OR `0.05 ≤ p_one_HAC ≤ 0.10`, but NOT both FAIL conditions. Action: routes to Clause-B HAC-bootstrap reconciliation in NB03 (spec §3.4) + convex-payoff escalation per spec §5.5 (quantile / GARCH-X / lower-tail). The convex-payoff arm tests directly whether the instrument is convex-fit even when mean-β is statistically ambiguous — load-bearing because Stage-2 M-sketch is a convex-payoff perpetual, not a linear-hedge contract.

The verdict-primitive printed in §2.8 is **PROVISIONAL** per spec §6 v1.0.2 CORRECTIONS-κ: NB03's R1 + R3 sign-AGREE check is load-bearing regardless of primary t-stat. If R1 OR R3 produces a sign-different β̂ in NB03, Clause-A FIRES even if primary is PASS.

**Lag-pattern concentration.** §2.7 prints `share_lag6 / share_lag9 / share_lag12` (each as `β_lag / β_composite`). The Pair D reference is `β_6 ≈ 80%` of composite (RC FLAG #3 — concentration in lag-6 means Pair D's effective signal is short-horizon-dominated). Possible profiles for this iteration:
- **Similar concentration (β_6 ≈ 70-90% of composite).** Confirms the Pair D RC-FLAG-#3 inheritance into Stage-2 dispatch brief; M-sketch hedge schedule should bias short-roll. Section J's structural transformation is short-horizon FX-arbitrage-driven (consistent with the BPO offshoring decision rhythm — quarterly contract negotiations, ~6-month implementation lag).
- **Different concentration (e.g., flat across lags or lag-12-dominant).** A NEW finding for this iteration vs Pair D. Section J ⊂ G–T; if Section J narrows the lag profile, the offshoring rhythm may be longer-cycle (12-month contract / fiscal-year-aligned decisions). M-sketch hedge schedule must shift accordingly. Phase-3 result memo §11.X(b) per spec §9.17(b) MUST report this primary-vs-Pair-D-inherited lag-pattern comparison verbatim.

**Composite-SE deflation empirical confirmation.** §2.6 prints `deflation_ratio = SE_composite / sqrt(SE_β_6² + SE_β_9² + SE_β_12²)`. Per spec §5.3 multicollinearity acknowledgment + Trio 4 NB01 expectation, the X_lag6/X_lag9/X_lag12 lags are correlated (autocorrelated COP/USD over 12-month windows produces large pairwise correlations). Under multicollinearity, the off-diagonal elements of `Σ̂` are negative (one lag's coefficient picks up part of another's variance), and the linear-restriction `c'Σ̂c = ΣΣ_ii + 2·Σ_off` is reduced (the 2× off-diagonal contribution is negative). Therefore `SE_composite < sqrt(ΣSE²)` — i.e., `deflation_ratio < 1.0` is the spec-anticipated regime. If §2.6 prints `deflation_ratio < 1.0`, the multicollinearity-deflation hypothesis is confirmed and the composite is more precisely identified than the per-lag coefficients individually. If `deflation_ratio ≥ 1.0`, this is anomalous (off-diagonal contributions are net-positive, meaning lags reinforce variance instead of canceling) — the printed warning would route to a re-fit sanity check.

**Pair D contrast detail.** Pair D's β_composite = +0.137 PASS verdict on broad-services Section G–T is the precedent baseline. This iteration's β_composite on Section J-narrow may be:
- **Similar magnitude (Δβ within ~30% of Pair D, e.g., 0.10-0.18).** Section J is broadly representative of the structural-transformation signal in G–T; the compositional-accounting overlap (§9.16) is empirically realized as numeric-near-equivalence. The R5 robustness arm (G–T minus J) — currently pre-authorized for v1.1 spec revision — would be load-bearing to confirm that the residual G–T-minus-J signal is independent (or that Section J carries the bulk of the G–T signal).
- **Materially different magnitude (e.g., 2× larger or 10× smaller).** Section J is a distinct ICT-specific signal vs the broad-services aggregate. If LARGER (β > 0.27): Section J narrows produce a sharper offshoring fingerprint (consistent with the dev-AI iteration narrative — US tech-firm offshoring is more concentrated in ICT vs broad services). If SMALLER (β < 0.05): the Section-J-narrow signal is weaker than the aggregate, suggesting the offshoring transmission is dispersed across many service sub-sectors.
Phase-3 result memo §11.X(b) per spec §9.17(b) MUST report the realized β-contrast verbatim regardless of magnitude.

**CORRECTIONS-κ binding (R1+R3 hedge load-bearing).** Per spec §6 v1.0.2 κ-tightening, the verdict-primitive emitted in this trio is **provisional**:
- **R1 hedge** (NB03 Trio 1): refits the OLS with a `dummy_post_2021` indicator (Marco-2018 boundary regime control). If R1 produces sign-different β̂ vs primary, Clause-A FIRES.
- **R3 hedge** (NB03 Trio 3): refits the OLS on raw share `Y_p_raw` instead of logit-transformed `Y_p_logit`. If R3 produces sign-different β̂ vs primary, Clause-A FIRES.
Both R1 + R3 are LOAD-BEARING per κ-tightening — meaning even a primary PASS verdict requires R1 + R3 sign-AGREE to fully clear Clause-A. This trio cannot pre-empt those checks; the verdict-primitive printed here is the PRIMARY input only.

**Flag accounting + anti-fishing discipline.** Per spec §3.3, expected outcomes are pre-registered: PASS at sign-positive direction is the prior under the Baumol → arbitrage → offshoring chain. FAIL or INDETERMINATE are pre-registered alternative outcomes that route to disposition or convex-payoff escalation respectively — they are NOT HALT triggers; they are spec-pre-pinned routing decisions. Anti-fishing discipline holds throughout: NO threshold tuning post-hoc (PASS / FAIL boundaries from §3.3 are pre-registered constants `t > 1.645`, `p_one < 0.05` strict / `p_one > 0.10` fail), NO post-hoc covariate addition (the OLS specification is verbatim per spec §5.3 with intercept + 3 lags only), NO sample restriction (the 134-month window is pinned per CORRECTIONS-α' commit `21beffade`). The next gate is NB03 Trios 1-3 (R1 + R2 + R3 robustness) followed by NB03 Trio 4 (Clause-A + Clause-B reconciliation + final verdict gate).

**Trio-checkpoint discipline.** This trio HALTS per `feedback_notebook_trio_checkpoint`. Orchestrator reviews the printed verdict-primitive + composite β̂ + HAC SE + p_one_HAC + lag-pattern shares + composite-SE deflation diagnostic + Pair D contrast; if numerical exceptions did not fire (singular matrix / N mismatch / HAC fit failure) the trio outcome is normal regardless of which verdict primitive printed. Trio 3 (verdict-gate verification + numeric-gate audit) dispatches next.
